# 05. Our computation and communication costs

This notebook measures what our Python implementation costs per phase, split by party: the user U and
one server $S_i$. Sec. VI and Tables III-IV of the paper report the authors' C++/NTL numbers at PARAM II.
We do not reproduce or quote them. Every number below is ours, at PARAM I (PARAM II is unsafe, see
notebook 06), with $N = 3$ servers, threshold $t = 2$ and 100 KB documents.

How each timing is taken:

- The kernel runs on one performance core, and the next cell prints the conditions.
- Each timed item gets one warmup run, then at least 7 timed runs. We print the median and the min-max
  range from `time.perf_counter`.
- Every run, the warmup included, checks its own result, and the cell prints the failure count.
- During the warmup a counting wrapper around `BPPRF.a_x` counts how many PRF chains $a_x$ each party
  ran. The timed runs run without it.
- Each run of a phase is followed by one timed $a_x$. A phase's time is then explained with the $a_x$
  taken alternately with it, under the same conditions.
- After each run the cell reads CPU 2's clock, and each item's table prints its median and range.

In [1]:
import os
import sys

numpy_before_pinning = "numpy" in sys.modules
os.sched_setaffinity(0, {2})

import importlib.metadata
import platform
from pathlib import Path


def cpu_file(name):
    try:
        return Path(f"/sys/devices/system/cpu/cpu2/{name}").read_text().strip()
    except OSError:
        return "unreadable"


def cpu2_mhz():
    try:
        return int(cpu_file("cpufreq/scaling_cur_freq")) // 1000
    except ValueError:
        return 0


def cpu_jiffies():
    rows = (line.split() for line in Path("/proc/stat").read_text().splitlines() if line.startswith("cpu") and line[3].isdigit())
    return {int(row[0][3:]): (sum(map(int, row[1:])), int(row[4]) + int(row[5])) for row in rows}


model = next(line.split(":", 1)[1].strip() for line in Path("/proc/cpuinfo").read_text().splitlines() if line.startswith("model name"))
load_before = os.getloadavg()
jiffies_before = cpu_jiffies()
print(f"CPU: {model}, {os.cpu_count()} logical CPUs")
print(f"kernel main thread affinity: {sorted(os.sched_getaffinity(0))} (numpy imported before pinning: {numpy_before_pinning})")
print(f"CPU 2: max {int(cpu_file('cpufreq/cpuinfo_max_freq')) // 1000} MHz, governor {cpu_file('cpufreq/scaling_governor')}, hyperthread siblings {cpu_file('topology/thread_siblings_list')}")
print(f"load average before (1, 5, 15 min): {load_before}")
print(f"Python {platform.python_version()}, numpy {importlib.metadata.version('numpy')}, "
      f"dilithium-py {importlib.metadata.version('dilithium-py')}, cryptography {importlib.metadata.version('cryptography')}")

CPU: 13th Gen Intel(R) Core(TM) i5-13500, 20 logical CPUs
kernel main thread affinity: [2] (numpy imported before pinning: False)
CPU 2: max 4800 MHz, governor powersave, hyperthread siblings 2-3
load average before (1, 5, 15 min): (0.974609375, 0.8212890625, 0.83544921875)
Python 3.14.8, numpy 2.5.3, dilithium-py 1.4.0, cryptography 50.0.2


## Setup and the measuring harness

`measure(trial)` calls `trial(clock)` once as a warmup that counts PRF chains and then `RUNS` times. A trial makes the
same party calls, in the same order, as the matching function in `protocol.py`, and wraps each call in
`clock(party, method, *args)`, which adds the call's time and minor page faults to that party. Work
outside `clock` (building a fresh `QPASEUser`, logging in before a timed Retrieve, the correctness checks)
is not timed. A trial returns whether its result was correct and the messages it sent, for the communication table at the end.

`setup_servers` runs the DKG and links the servers for the key update. alice registers and logs in once
here. $\mu = 1000$ leaves room for every Login below, since L1 counts each attempt.

In [2]:
import cProfile
import pstats
import resource
import statistics
import time
from collections import defaultdict
from contextlib import contextmanager
from dataclasses import dataclass

import numpy as np

from ris_sem_project import protocol
from ris_sem_project.display import payload_size, table
from ris_sem_project.params import PARAM_I, Lattice, setup
from ris_sem_project.parties import RetrieveResponse
from ris_sem_project.parties.qpase_server import ShareOutcome
from ris_sem_project.parties.qpase_user import QPASEUser
from ris_sem_project.primitives import toprf
from ris_sem_project.primitives.bp_prf import BPPRF, encode_input
from ris_sem_project.primitives.encoding import pack
from ris_sem_project.primitives.gadget import g_inverse
from ris_sem_project.primitives.linear_hash import linear_hash
from ris_sem_project.primitives.sampling import TAIL, gaussian, uniform
from ris_sem_project.primitives.secret_sharing import reconstruct
from ris_sem_project.primitives.signature import SEED_SIZE, keypair_from_seed, sign, verify
from ris_sem_project.primitives.symmetric import KEY_SIZE, decrypt, encrypt

RUNS = 7
DOC_SIZE = 100_000
PASSWORD = "correct horse battery staple"


class Clock:
    def __init__(self):
        self.seconds = defaultdict(float)
        self.faults = defaultdict(int)
        self.chains = defaultdict(int)
        self.party = None

    def __call__(self, party, method, *args):
        self.party = party
        faults = resource.getrusage(resource.RUSAGE_SELF).ru_minflt
        start = time.perf_counter()
        result = method(*args)
        self.seconds[party] += time.perf_counter() - start
        self.faults[party] += resource.getrusage(resource.RUSAGE_SELF).ru_minflt - faults
        self.party = None
        return result


@contextmanager
def counting_chains(clock):
    a_x = BPPRF.a_x

    def counted(prf, x):
        if clock.party is not None:
            clock.chains[clock.party] += 1
        return a_x(prf, x)

    BPPRF.a_x = counted
    try:
        yield
    finally:
        BPPRF.a_x = a_x


@dataclass
class Measured:
    runs: list
    faults: list
    failures: int
    chains: dict
    wire: list
    mhz: list

    def samples(self, party):
        return [run[party] for run in self.runs]

    def median(self, party):
        return statistics.median(self.samples(party))

    def mean(self, party):
        return statistics.mean(self.samples(party))

    def median_faults(self, party):
        return statistics.median(run[party] for run in self.faults)


def measure(trial, runs=RUNS):
    warmup = Clock()
    with counting_chains(warmup):
        ok, _ = trial(warmup)
    chains = {party: warmup.chains[party] for party in warmup.seconds}
    failures = int(not ok)
    timed, faults, mhz = [], [], []
    for _ in range(runs):
        clock = Clock()
        ok, wire = trial(clock)
        mhz.append(cpu2_mhz())
        failures += not ok
        timed.append(dict(clock.seconds))
        faults.append(dict(clock.faults))
    return Measured(runs=timed, faults=faults, failures=failures, chains=chains, wire=wire, mhz=mhz)


def duration(seconds):
    if seconds >= 1:
        return f"{seconds:.2f} s"
    ms = seconds * 1e3
    return f"{ms:.0f} ms" if ms >= 100 else f"{ms:.1f} ms" if ms >= 10 else f"{ms:.2f} ms" if ms >= 0.1 else f"{ms:.3f} ms"


def in_ms(seconds):
    return f"{seconds * 1e3:.0f} ms" if seconds >= 0.01 else f"{seconds * 1e3:.2f} ms"


def spread(samples):
    return f"{duration(min(samples))} to {duration(max(samples))}"


def report(measured, parties, name):
    rows = [(p, duration(statistics.median(measured.samples(p))), spread(measured.samples(p)), len(measured.runs)) for p in parties]
    print(table(("party", "median", "range (min to max)", "runs"), rows))
    print(f"{name} failures: {measured.failures} of {len(measured.runs) + 1} runs (warmup included)")
    print(f"CPU 2 clock read after each run: median {statistics.median(measured.mhz)} MHz, {min(measured.mhz)} to {max(measured.mhz)} MHz")


pp = setup(PARAM_I, N=3, t=2, mu=1000)
lattice = Lattice.from_pp(pp)
ring = lattice.ring
servers = protocol.setup_servers(pp)
alice = QPASEUser(pp, "alice", PASSWORD)
alice_registered = protocol.register(alice, servers)
protocol.login(alice, servers, (1, 2))
print(f"{PARAM_I.name}: q = {pp.q}, n = {pp.n}, ell = {pp.ell}, L = {pp.L}; N = {pp.N}, t = {pp.t}, mu = {pp.mu}")
print(f"PRF key, K_u and dsk: ell * n = {ring.ell * ring.n} bits = {ring.ell * ring.n // 8} B; documents: {DOC_SIZE} B")

PARAM I: q = 268435399, n = 512, ell = 28, L = 128; N = 3, t = 2, mu = 1000
PRF key, K_u and dsk: ell * n = 14336 bits = 1792 B; documents: 100000 B


## Building blocks

**One PRF chain $a_x$.** $a_x = a_{x_1} G^{-1}(a_{x_2} G^{-1}(\cdots G^{-1}(a_{x_L})))$ costs $L - 1 = 127$
steps, and each step needs the previous one's output. The check compares every run with
$a_{x_1} G^{-1}(a_{x_2 \dots x_L})$ recomputed once through `Ring.mul`, a different code path from the
`einsum` the chain uses. One more run under `cProfile`, not timed, shows where one chain's time goes.

Besides the work itself, two things move this number. One chain allocates a few MB of temporary arrays
per step (6.4 MB for one $G^{-1}(v)$ spectrum), so the allocator's state decides how many minor page faults a run pays. And the core's clock
changes under the `powersave` governor. Every timed item records both, the faults per run and CPU 2's
clock read from cpufreq right after each run. Because both can drift during the notebook, each phase
below is explained with an $a_x$ timed alongside it, not with this one.

In [3]:
x = encode_input(PASSWORD.encode(), pp.L)
shape = (ring.ell, ring.n)
a_bits = (ring.from_bytes(pp.a0, shape), ring.from_bytes(pp.a1, shape))
a_rest = lattice.prf.a_x(x[1:])
a_x_reference = ring.mul(a_bits[x[0]][:, None, :], g_inverse(ring, a_rest).astype(np.int64)).sum(axis=0) % ring.q


def a_x_trial(clock):
    return np.array_equal(clock("a_x", lattice.prf.a_x, x), a_x_reference), []


chain = measure(a_x_trial)
report(chain, ["a_x"], "a_x")
A_X = chain.median("a_x")
print(f"one PRF chain a_x: median {in_ms(A_X)} over {RUNS} runs, {duration(A_X / (pp.L - 1))} per step over {pp.L - 1} steps")
print(f"minor page faults per a_x run: median {chain.median_faults('a_x'):.0f}")
print()
profile = cProfile.Profile()
profile.runcall(lattice.prf.a_x, x)
stats = pstats.Stats(profile).stats
total = sum(tottime for _, _, tottime, _, _ in stats.values())
hot = sorted(((tottime, f"{Path(file).name}:{name}" if file != "~" else name) for (file, _, name), (_, _, tottime, _, _) in stats.items()), reverse=True)[:5]
print("where one a_x spends its time (one extra run under cProfile, not timed, own time per function):")
print(table(("function", "share"), [(name, f"{tottime / total:.0%}") for tottime, name in hot]))

party  median  range (min to max)  runs
-----  ------  ------------------  ----
a_x    494 ms  494 ms to 502 ms    7
a_x failures: 0 of 8 runs (warmup included)
CPU 2 clock read after each run: median 3900 MHz, 3778 to 3900 MHz
one PRF chain a_x: median 494 ms over 7 runs, 3.89 ms per step over 127 steps
minor page faults per a_x run: median 3510



where one a_x spends its time (one extra run under cProfile, not timed, own time per function):
function                                                  share
--------------------------------------------------------  -----
<built-in method numpy._core._multiarray_umath.c_einsum>  36%
_pocketfft.py:_raw_fft                                    33%
ring.py:spectrum                                          15%
gadget.py:g_inverse                                       8%
<method 'astype' of 'numpy.ndarray' objects>              3%


**The other building blocks.** `Ring.mul` multiplies an $R_q^{1 \times \ell}$ row by one ring element, the
product inside every PRF output and TOPRF message. It is checked against a schoolbook negacyclic
convolution in exact int64 arithmetic. ML-DSA-44 is checked by verifying a signature and rejecting a
changed message. AES-256-GCM is checked by decrypting back to the 100 KB document. The linear hash $H$ of
the key update is checked for linearity. The TOPRF steps are checked by the protocol property: the
combined $F$, extracted with alice's stored hint $\sigma$, gives alice's registered $K_u$. The blinded $b$ is
also checked to be $a r + a_x + 2e$ with $2e$ even and inside the sampler's support.

In [4]:
r_mul = gaussian(ring, (ring.n,), pp.sigma)


def negacyclic_schoolbook(row, small):
    full = np.convolve(row, small)
    return (full[: ring.n] - np.append(full[ring.n :], 0)) % ring.q


mul_reference = np.array([negacyclic_schoolbook(row, ring.centered(r_mul)) for row in lattice.a])
a_x_pw = lattice.prf.a_x(x)
keys = {i: servers[i].key.k_i for i in (1, 2)}
server_pks = {i: servers[i].key.pk_i for i in (1, 2)}
alice_hint = ring.unpack_bits(servers[1].users["alice"].hint)


def mul_trial(clock):
    return np.array_equal(clock("Ring.mul (ell, n) x (n,)", ring.mul, lattice.a, r_mul), mul_reference), []


def mldsa_trial(clock):
    pk, sk = clock("ML-DSA-44 keygen", keypair_from_seed, os.urandom(SEED_SIZE))
    message = pack(b"alice", os.urandom(32))
    signature = clock("ML-DSA-44 sign", sign, sk, message)
    accepted = clock("ML-DSA-44 verify", verify, pk, message, signature)
    return accepted and not verify(pk, message + b"!", signature), []


def aes_trial(clock):
    key, document = os.urandom(KEY_SIZE), os.urandom(DOC_SIZE)
    blob = clock("AES-256-GCM encrypt 100 KB", encrypt, key, document)
    opened = clock("AES-256-GCM decrypt 100 KB", decrypt, key, blob)
    return opened == document and len(blob) == DOC_SIZE + 28, []


def hash_trial(clock):
    u, w = uniform(ring, (ring.n,)), uniform(ring, (ring.n,))
    hu = clock("linear hash H", linear_hash, ring, u)
    return np.array_equal(ring.add(hu, linear_hash(ring, w)), linear_hash(ring, ring.add(u, w))), []


def toprf_trial(clock):
    b, r = clock("TOPRF blind (U)", toprf.blind, ring, lattice.a, a_x_pw, pp.sigma)
    evaluations = {1: clock("TOPRF evaluate (S_i)", toprf.evaluate, ring, b, keys[1], pp.sigma), 2: toprf.evaluate(ring, b, keys[2], pp.sigma)}
    F = clock("TOPRF combine (U)", toprf.combine, ring, evaluations, server_pks, r)
    noise = ring.centered(b - ring.mul(lattice.a, r) - a_x_pw)
    blind_ok = bool((noise % 2 == 0).all() and np.abs(noise).max() <= 2 * np.ceil(TAIL * pp.sigma))
    return blind_ok and toprf.finalize(ring, F, alice_hint)[0] == alice_registered.k_u, []


blocks = {
    "Ring.mul": (measure(mul_trial, 31), ["Ring.mul (ell, n) x (n,)"]),
    "ML-DSA-44": (measure(mldsa_trial, 101), ["ML-DSA-44 keygen", "ML-DSA-44 sign", "ML-DSA-44 verify"]),
    "AES-256-GCM": (measure(aes_trial, 31), ["AES-256-GCM encrypt 100 KB", "AES-256-GCM decrypt 100 KB"]),
    "H": (measure(hash_trial, 31), ["linear hash H"]),
    "TOPRF": (measure(toprf_trial), ["TOPRF blind (U)", "TOPRF evaluate (S_i)", "TOPRF combine (U)"]),
}
rows = []
for name, (measured, parts) in blocks.items():
    rows += [
        (part, duration(measured.median(part)), spread(measured.samples(part)), duration(measured.mean(part)), len(measured.runs), measured.failures)
        for part in parts
    ]
print(table(("building block", "median", "range (min to max)", "mean", "runs", "failures"), rows))
UNIT = {
    "mul": blocks["Ring.mul"][0].mean("Ring.mul (ell, n) x (n,)"),
    "keygen": blocks["ML-DSA-44"][0].mean("ML-DSA-44 keygen"),
    "sign": blocks["ML-DSA-44"][0].mean("ML-DSA-44 sign"),
    "verify": blocks["ML-DSA-44"][0].mean("ML-DSA-44 verify"),
    "aes_enc": blocks["AES-256-GCM"][0].mean("AES-256-GCM encrypt 100 KB"),
    "aes_dec": blocks["AES-256-GCM"][0].mean("AES-256-GCM decrypt 100 KB"),
    "hash": blocks["H"][0].mean("linear hash H"),
    "blind": blocks["TOPRF"][0].mean("TOPRF blind (U)"),
    "evaluate": blocks["TOPRF"][0].mean("TOPRF evaluate (S_i)"),
    "combine": blocks["TOPRF"][0].mean("TOPRF combine (U)"),
}
print()
for part in ("ML-DSA-44 sign", "ML-DSA-44 verify", "Ring.mul (ell, n) x (n,)"):
    measured = next(m for m, parts in blocks.values() if part in parts)
    print(f"{part}: median {in_ms(measured.median(part))} over {len(measured.runs)} runs")
print(f"building block failures: {sum(m.failures for m, _ in blocks.values())}")

building block              median    range (min to max)    mean      runs  failures
--------------------------  --------  --------------------  --------  ----  --------
Ring.mul (ell, n) x (n,)    0.67 ms   0.66 ms to 0.79 ms    0.69 ms   31    0
ML-DSA-44 keygen            3.97 ms   3.91 ms to 4.75 ms    3.99 ms   101   0
ML-DSA-44 sign              20.5 ms   10.1 ms to 110 ms     27.0 ms   101   0
ML-DSA-44 verify            4.81 ms   4.73 ms to 5.53 ms    4.82 ms   101   0
AES-256-GCM encrypt 100 KB  0.021 ms  0.021 ms to 0.036 ms  0.022 ms  31    0
AES-256-GCM decrypt 100 KB  0.021 ms  0.021 ms to 0.043 ms  0.022 ms  31    0
linear hash H               0.21 ms   0.20 ms to 0.21 ms    0.21 ms   31    0
TOPRF blind (U)             1.15 ms   1.14 ms to 1.19 ms    1.16 ms   7     0
TOPRF evaluate (S_i)        1.09 ms   1.08 ms to 1.10 ms    1.09 ms   7     0
TOPRF combine (U)           1.07 ms   1.04 ms to 1.07 ms    1.06 ms   7     0

ML-DSA-44 sign: median 20 ms over 101 runs
ML-DSA

## The phases, split by party

Each trial below follows its `protocol.py` function call by call and charges each call to U or to the
server that ran it. The tables show U and $S_1$. Before each phase's table, the cell states what one run
checks.

**Register** (Sec. IV-B) with all $N$ servers, one new user per run. Check: every server stores the same
$pk_u$, and a Login of that user afterwards (not timed) reproduces $K_u$ and $pk_u$.

In [5]:
def alternating(trial):
    def trial_then_a_x(clock):
        ok, wire = trial(clock)
        return ok and a_x_trial(clock)[0], wire

    return trial_then_a_x


register_ids = iter(f"user-{k}" for k in range(100))


def register_trial(clock):
    user = QPASEUser(pp, next(register_ids), PASSWORD)
    for i, server in servers.items():
        clock(f"S_{i}", server.register_user, user.user_id)
    requests = clock("U", user.toprf_request, sorted(servers))
    responses = {i: clock(f"S_{i}", servers[i].toprf_evaluate, request) for i, request in requests.items()}
    k_u = clock("U", user.toprf_finalize, list(responses.values()))
    key_message = clock("U", user.register_keygen, k_u)
    for i, server in servers.items():
        clock(f"S_{i}", server.register_store_key, key_message)
    later = protocol.login(QPASEUser(pp, user.user_id, PASSWORD), servers, (1, 2))
    stored = {server.users[user.user_id].user_pk for server in servers.values()}
    ok = stored == {key_message.user_pk} and later.k_u == k_u and later.user_pk == key_message.user_pk
    wire = [
        ("ID_u (duplicate check)", "U -> S", [user.user_id] * pp.N),
        ("TOPRFRequest", "U -> S", list(requests.values())),
        ("TOPRFResponse", "S -> U", list(responses.values())),
        ("RegisterKeyMessage", "U -> S", [key_message] * pp.N),
    ]
    return ok, wire


register = measure(alternating(register_trial))
report(register, ["U", "S_1", "a_x"], "Register")
print(f"Register at U: median {in_ms(register.median('U'))}; at S_1: median {in_ms(register.median('S_1'))}")

party  median   range (min to max)  runs
-----  -------  ------------------  ----
U      1.43 s   1.40 s to 1.45 s    7
S_1    1.21 ms  1.18 ms to 1.23 ms  7
a_x    500 ms   467 ms to 506 ms    7
Register failures: 0 of 8 runs (warmup included)
CPU 2 clock read after each run: median 3929 MHz, 3774 to 4000 MHz
Register at U: median 1434 ms; at S_1: median 1.21 ms


**Login** (Sec. IV-C) of alice with $\{S_1, S_2\}$, a fresh `QPASEUser` holding only the password each
run. Check: $K_u$ and $pk_u$ equal alice's registered ones, every contacted server returned that $pk_u$,
and both servers accepted $sig_u$ in L3.

In [6]:
def login_trial(clock):
    user = QPASEUser(pp, "alice", PASSWORD)
    requests = clock("U", user.toprf_request, (1, 2))
    responses = {i: clock(f"S_{i}", servers[i].toprf_evaluate, request) for i, request in requests.items()}
    user_pk = responses[1].user_pk
    k_u = clock("U", user.toprf_finalize, list(responses.values()))
    sids, proofs, accepted = {}, {}, {}
    for i in requests:
        sids[i] = clock(f"S_{i}", servers[i].new_session, user.user_id)
        proofs[i] = clock("U", user.login_verify, k_u, user_pk, sids[i])
        accepted[i] = clock(f"S_{i}", servers[i].login_verify, proofs[i])
    ok = (
        k_u == alice_registered.k_u
        and all(response.user_pk == alice_registered.key_message.user_pk for response in responses.values())
        and all(accepted.values())
    )
    wire = [
        ("TOPRFRequest", "U -> S", list(requests.values())),
        ("TOPRFResponse (with pk_u, sigma)", "S -> U", list(responses.values())),
        ("sid", "S -> U", list(sids.values())),
        ("LoginProof", "U -> S", list(proofs.values())),
    ]
    return ok, wire


login = measure(alternating(login_trial))
report(login, ["U", "S_1", "a_x"], "Login")
print(f"Login at U: median {in_ms(login.median('U'))}; at S_1: median {in_ms(login.median('S_1'))}")

party  median   range (min to max)  runs
-----  -------  ------------------  ----
U      1.94 s   1.91 s to 1.97 s    7
S_1    6.07 ms  5.95 ms to 6.41 ms  7
a_x    465 ms   462 ms to 477 ms    7
Login failures: 0 of 8 runs (warmup included)
CPU 2 clock read after each run: median 4000 MHz, 3878 to 4000 MHz
Login at U: median 1943 ms; at S_1: median 6.07 ms


**Outsource** (Sec. IV-D) of one fresh 100 KB document to $S_1$ by the logged-in alice. U caches each
$dsk$ for the session, so a keyword's first document pays for $dsk = HKDF(K_u, w)$ and later ones do not.
Each run uses a new keyword, so every run pays for it. Check: $S_1$ accepted $sig_{Ct}$ and stored the item,
and alice's own R3 check (not timed) opens it to the original document.

In [7]:
outsource_ids = iter(range(1000, 2000))


def outsource_trial(clock):
    item_id = next(outsource_ids)
    keyword, document = f"keyword-{item_id}", os.urandom(DOC_SIZE)
    request = clock("U", alice.outsource, keyword, document, item_id)
    stored = clock("S_1", servers[1].outsource_store, request)
    opened = alice.retrieve_open([keyword], [RetrieveResponse(server_id=1, results=(request.item,))])
    ok = stored and servers[1].storage["alice"][-1] == request.item and [o.data for o in opened] == [document]
    return ok, [("OutsourceRequest", "U -> S", [request])]


outsource = measure(alternating(outsource_trial))
report(outsource, ["U", "S_1", "a_x"], "Outsource")
print(f"Outsource of {DOC_SIZE} B at U: median {in_ms(outsource.median('U'))}; at S_1: median {in_ms(outsource.median('S_1'))}")

party  median   range (min to max)  runs
-----  -------  ------------------  ----
U      966 ms   955 ms to 996 ms    7
S_1    5.01 ms  4.97 ms to 5.12 ms  7
a_x    459 ms   458 ms to 465 ms    7
Outsource failures: 0 of 8 runs (warmup included)
CPU 2 clock read after each run: median 4000 MHz, 3999 to 4000 MHz
Outsource of 100000 B at U: median 966 ms; at S_1: median 5.01 ms


**Retrieve** (Sec. IV-E). bob stores 10 documents of 100 KB on $S_1$, 2 under `medical` and 8 under four
other keywords, and none on $S_2, S_3$. R1 goes to all $N$ servers. Each run starts from a fresh Login
(not timed), so R1 derives $dsk$ again as a first query of the session does. Check: $S_1$'s scan stops at
$Dec$ for the 8 other items and matches the 2 `medical` ones, $L_1$ holds exactly those, and R3 returns
the 2 original documents.

In [8]:
bob = QPASEUser(pp, "bob", "bob's own password")
protocol.register(bob, servers)
protocol.login(bob, servers, (1, 2))
QUERY = "medical"
bob_items = [(2000 + k, QUERY if k in (3, 7) else ("travel", "invoice", "meeting", "tax")[k % 4], os.urandom(DOC_SIZE)) for k in range(10)]
for item_id, keyword, document in bob_items:
    protocol.outsource(bob, servers[1], keyword, document, item_id)
expected = [(item_id, document) for item_id, keyword, document in bob_items if keyword == QUERY]
print(f"S_1 holds {len(servers[1].storage['bob'])} items for bob, {len(expected)} under {QUERY!r}; S_2, S_3 hold {len(servers[2].storage.get('bob', []))}, {len(servers[3].storage.get('bob', []))}")


def retrieve_trial(clock):
    user = QPASEUser(pp, "bob", "bob's own password")
    protocol.login(user, servers, (1, 2))
    request = clock("U", user.retrieve_request, [QUERY])
    scans = {i: clock(f"S_{i}", server.retrieve_scan, request) for i, server in servers.items()}
    opened = clock("U", user.retrieve_open, [QUERY], [scan.response for scan in scans.values()])
    outcomes = sorted(s.outcome.name for s in scans[1].items)
    ok = (
        outcomes == sorted(["DECRYPT_FAILED"] * 8 + ["MATCH"] * 2)
        and [item.item_id for item in scans[1].response.results] == [item_id for item_id, _ in expected]
        and [(o.item_id, o.data) for o in opened] == expected
    )
    wire = [
        ("RetrieveRequest", "U -> S", [request] * pp.N),
        ("RetrieveResponse (L_i)", "S -> U", [scan.response for scan in scans.values()]),
    ]
    return ok, wire


retrieve = measure(alternating(retrieve_trial))
report(retrieve, ["U", "S_1", "S_2", "a_x"], "Retrieve")
print(f"Retrieve at U: median {in_ms(retrieve.median('U'))}; at S_1 (10 items, 2 match): median {in_ms(retrieve.median('S_1'))}")

S_1 holds 10 items for bob, 2 under 'medical'; S_2, S_3 hold 0, 0


party  median   range (min to max)  runs
-----  -------  ------------------  ----
U      1.52 s   1.51 s to 1.54 s    7
S_1    1.01 s   1.01 s to 1.02 s    7
S_2    4.99 ms  4.95 ms to 5.08 ms  7
a_x    495 ms   492 ms to 497 ms    7
Retrieve failures: 0 of 8 runs (warmup included)
CPU 2 clock read after each run: median 3837 MHz, 3765 to 3900 MHz
Retrieve at U: median 1523 ms; at S_1 (10 items, 2 match): median 1010 ms


**Server key update** (Sec. IV-F), one round among all $N$ servers per run, in the order of
`protocol.key_update`: every $S_j$ deals its shares (steps 1-3), every $S_i$ checks the $N$ shares it
received (step 4), then every $S_i$ applies them (steps 4-5). `key_update_apply` opens and checks each
share again, so step 4 runs twice per share. The table splits $S_1$ by step and adds the whole round
(all $N$ servers, one after another). Check: all $N^2$ shares ACCEPTED, every epoch moved up by one, and
the master key $K = \sum_{i \in \{1,2\}} \lambda_i k_i$ is unchanged. After the last round alice logs in
(not timed) and must get her registered $K_u$.

In [9]:
def key_update_trial(clock):
    K_before = reconstruct(ring, {i: servers[i].key.k_i for i in (1, 2)})
    epochs = [server.epoch for server in servers.values()]
    dealt = {j: clock(f"S_{j} deal", server.key_update_shares) for j, server in servers.items()}
    received = {i: [dealt[j][i] for j in servers] for i in servers}
    checks = [c for i, server in servers.items() for c in clock(f"S_{i} check", server.key_update_check, received[i])]
    for i, server in servers.items():
        clock(f"S_{i} apply", server.key_update_apply, received[i])
    ok = (
        len(checks) == pp.N**2
        and all(c.outcome is ShareOutcome.ACCEPTED for c in checks)
        and [server.epoch for server in servers.values()] == [e + 1 for e in epochs]
        and np.array_equal(reconstruct(ring, {i: servers[i].key.k_i for i in (1, 2)}), K_before)
    )
    network = [dealt[j][i] for j in servers for i in servers if i != j]
    local = [dealt[j][j] for j in servers]
    return ok, [("KeyUpdateShare", "S -> S", network), ("KeyUpdateShare to itself", "local", local)]


key_update = measure(key_update_trial, 15)
after_updates = protocol.login(QPASEUser(pp, "alice", PASSWORD), servers, (1, 2))
key_update.failures += after_updates.k_u != alice_registered.k_u
for run in key_update.runs:
    run["S_1"] = sum(run[f"S_1 {step}"] for step in ("deal", "check", "apply"))
    run["round"] = sum(seconds for label, seconds in run.items() if label != "S_1")
report(key_update, ["S_1 deal", "S_1 check", "S_1 apply", "S_1", "round"], "Key update")
print(f"epochs now {[s.epoch for s in servers.values()]}; alice's K_u after {len(key_update.runs) + 1} updates equals Register's: {after_updates.k_u == alice_registered.k_u}")
print(f"Key update at S_1: median {in_ms(key_update.median('S_1'))}; whole round, N = {pp.N}: median {in_ms(key_update.median('round'))}")

party      median   range (min to max)  runs
---------  -------  ------------------  ----
S_1 deal   63.0 ms  33.9 ms to 109 ms   15
S_1 check  15.2 ms  15.1 ms to 15.4 ms  15
S_1 apply  16.4 ms  16.3 ms to 17.1 ms  15
S_1        94.7 ms  65.7 ms to 141 ms   15
round      310 ms   229 ms to 392 ms    15
Key update failures: 0 of 16 runs (warmup included)
CPU 2 clock read after each run: median 3799 MHz, 3725 to 4000 MHz
epochs now [16, 16, 16]; alice's K_u after 16 updates equals Register's: True
Key update at S_1: median 95 ms; whole round, N = 3: median 310 ms


## Explain the number

**Where the PRF chain counts come from.** Every PRF output (`prf.prf`, through `BPPRF.evaluate`) runs one
chain $a_x$ and one `Ring.mul`. `Hkdf(K_u)` runs one PRF to extract, and each `Hkdf.derive` one more.
The counts below are read from the code path. The table then compares them with the `a_x` calls the
wrapper counted in each party's warmup run.

| phase | party | chains | where they run |
| --- | --- | --- | --- |
| Register | U | 3 | `toprf_request`: $a_x$ of the password. `register_keygen`: `Hkdf(K_u)` extract, then derive the ML-DSA seed. |
| Register | $S_i$ | 0 | `toprf_evaluate` is $b k_i + 2e'_i$, one `Ring.mul`. |
| Login | U | 4 | `toprf_request`: $a_x$ of the password. The first `login_verify`: `Hkdf(K_u)` extract, then derive the ML-DSA seed and the AES key. The second `login_verify` reuses the session. |
| Login | $S_i$ | 0 | `toprf_evaluate`, then `Ver` of $sig_u$ in `login_verify`. |
| Outsource | U | 2 | `search_key` derives $dsk$ (new keyword each run). `build_item`: $v = PRF(dsk, \rho)$. |
| Outsource | $S_i$ | 0 | `Ver` of $sig_{Ct}$. |
| Retrieve | U | 3 | R1 derives $dsk$ (fresh session). R3 recomputes $v$ for each of the 2 returned items. |
| Retrieve | $S_1$ | 2 | `scan_item` per stored item. The 8 other items stop at $Dec(dsk, Ct)$, before the PRF. Each of the 2 matches runs `Ver` and one PRF. |
| Key update | $S_i$ | 0 | $H$ is a matrix product and $E(i, \cdot)$ is AES. No PRF. |

**The cost model.** Each party's other work is counted the same way, in calls of the building blocks
timed above: `Ring.mul` outside a PRF chain, TOPRF blind / evaluate / combine, ML-DSA-44 keygen / sign /
verify, AES-256-GCM on 100 KB, and $H$. The model is $\sum_\text{op} \text{count} \times \text{cost}$. The cost of
a chain is the median of the $a_x$ timed alternately with the phase. Every other cost is the building
block's mean, since a sum of calls tracks the mean: ML-DSA-44 signing repeats a rejection loop a random
number of times, so its mean is above its median. The share column is the median over runs of (chains $\times$ that run's
$a_x$) / (that run's phase time). The limiter is the model's largest term.

In [10]:
OPS = {
    ("Register", "U"): dict(chain=3, blind=1, combine=1, mul=2, keygen=1),
    ("Register", "S_1"): dict(evaluate=1),
    ("Login", "U"): dict(chain=4, blind=1, combine=1, mul=3, keygen=1, sign=2),
    ("Login", "S_1"): dict(evaluate=1, verify=1),
    ("Outsource", "U"): dict(chain=2, mul=2, sign=2, aes_enc=1),
    ("Outsource", "S_1"): dict(verify=1),
    ("Retrieve", "U"): dict(chain=3, mul=3, sign=1, verify=2, aes_dec=2),
    ("Retrieve", "S_1"): dict(chain=2, mul=2, verify=3),
    ("Key update", "S_1"): dict(sign=3, verify=6, hash=7, mul=1),
    ("Key update", "round"): dict(sign=9, verify=18, hash=21, mul=3),
}
PHASES = {"Register": register, "Login": login, "Outsource": outsource, "Retrieve": retrieve, "Key update": key_update}
LIMITER_NAME = {
    "chain": "PRF chains a_x", "sign": "ML-DSA-44 sign", "verify": "ML-DSA-44 verify",
    "evaluate": "TOPRF evaluate", "mul": "Ring.mul", "hash": "linear hash H",
}


def counted_chains(measured, party):
    if party == "round":
        return sum(measured.chains.values())
    if party == "S_1" and "S_1 deal" in measured.chains:
        return sum(measured.chains[f"S_1 {step}"] for step in ("deal", "check", "apply"))
    return measured.chains[party]


def phase_a_x(phase):
    measured = PHASES[phase]
    return measured.median("a_x") if "a_x" in measured.runs[0] else A_X


timing_rows, explain_rows, lines, limiters, counted = [], [], [], {}, {}
for (phase, party), ops in OPS.items():
    measured = PHASES[phase]
    median = measured.median(party)
    a_x = phase_a_x(phase)
    terms = {op: count * (a_x if op == "chain" else UNIT[op]) for op, count in ops.items()}
    model = sum(terms.values())
    limiter = max(terms, key=terms.get)
    limiters[(phase, party)] = limiter
    counted[(phase, party)] = (ops.get("chain", 0), counted_chains(measured, party))
    chains = ops.get("chain", 0)
    paired = statistics.median(chains * run["a_x"] / run[party] for run in measured.runs) if "a_x" in measured.runs[0] else 0
    timing_rows.append((phase, party, duration(median), spread(measured.samples(party)), len(measured.runs), measured.failures))
    explain_rows.append((
        phase, party, f"{chains} / {counted[(phase, party)][1]}", duration(a_x), duration(chains * a_x) if chains else "0",
        f"{paired:.0%}", duration(model), f"{model / median:.2f}", f"{LIMITER_NAME[limiter]} ({terms[limiter] / median:.0%})",
    ))
    lines.append(
        f"{phase} at {party}: {chains} PRF chains, share in chains {paired:.0%}, limiter {LIMITER_NAME[limiter]} ({terms[limiter] / median:.0%}); "
        f"model {duration(model)} against the measured {duration(median)}"
    )
print(table(("phase", "party", "median", "range (min to max)", "runs", "failures"), timing_rows))
print()
print(table(
    ("phase", "party", "chains (code / counted)", "a_x", "chains x a_x", "share in chains", "model", "model / measured", "limiter (share of measured)"),
    explain_rows,
))
print()
print("\n".join(lines))

phase       party  median   range (min to max)  runs  failures
----------  -----  -------  ------------------  ----  --------
Register    U      1.43 s   1.40 s to 1.45 s    7     0
Register    S_1    1.21 ms  1.18 ms to 1.23 ms  7     0
Login       U      1.94 s   1.91 s to 1.97 s    7     0
Login       S_1    6.07 ms  5.95 ms to 6.41 ms  7     0
Outsource   U      966 ms   955 ms to 996 ms    7     0
Outsource   S_1    5.01 ms  4.97 ms to 5.12 ms  7     0
Retrieve    U      1.52 s   1.51 s to 1.54 s    7     0
Retrieve    S_1    1.01 s   1.01 s to 1.02 s    7     0
Key update  S_1    94.7 ms  65.7 ms to 141 ms   15    0
Key update  round  310 ms   229 ms to 392 ms    15    0

phase       party  chains (code / counted)  a_x     chains x a_x  share in chains  model    model / measured  limiter (share of measured)
----------  -----  -----------------------  ------  ------------  ---------------  -------  ----------------  ---------------------------
Register    U      3 / 3             

**The limiter of each phase.** The table above prints each phase's share, model and limiter, and the
checks at the end fail if the limiter differs from the one named here.

- **Register, U.** Its 3 PRF chains, the password's $a_x$ and the two HKDF calls behind the ML-DSA seed, take almost all of the time.
- **Register, $S_i$.** One TOPRF evaluation, $b k_i + 2e'_i$: one `Ring.mul` plus fresh Gaussian noise. There is no chain and no signature.
- **Login, U.** Its 4 PRF chains, the password's $a_x$ and three HKDF calls (extract, seed, AES key). The two signatures are a few percent.
- **Login, $S_i$.** The ML-DSA-44 verification of $sig_u$, a few times the cost of its TOPRF evaluation.
- **Outsource, U.** Its 2 PRF chains, for $dsk$ and $v$. AES over 100 KB and the two signatures barely show.
- **Outsource, $S_i$.** The ML-DSA-44 verification of $sig_{Ct}$. Hashing the 100 KB message inside it is cheap.
- **Retrieve, U.** Its 3 PRF chains, for $dsk$ in R1 and $v$ of the 2 returned items in R3.
- **Retrieve, $S_1$.** Its 2 PRF chains, one per matching item. The 8 other items fail at AES-GCM's tag check and cost almost nothing.
- **Key update.** ML-DSA-44 signing of the $N$ shares each server deals, then the $2N$ verifications, since step 4 runs once in `key_update_check` and again in `key_update_apply`.

**Why not faster.** In each U phase and in Retrieve at $S_1$ the chain is the limiter, and it cannot be
batched within one chain. Each of its 127 steps needs the previous step's output. Inside a step the
profile above puts most of the time in the forward FFT of the $\ell^2 = 784$ polynomials of
$G^{-1}(v)$ and in the `einsum` that contracts them with $a_0$ or $a_1$, both on one core. The kernel is
pinned to that core, so the numbers are single-core numbers, taken below the core's maximum clock (the
first cell prints the maximum, each item's table the clock read after its runs). The server phases without a chain are
bound by `dilithium-py`, a pure-Python ML-DSA-44.

## Communication

These counts are exact and not timed. `payload_size` adds up the bytes and str fields of a message,
recursing into nested dataclasses. ID_u counts as its UTF-8 bytes (`user-7` is 6 B, `alice` 5 B and
`bob` 3 B). Ints such as server ids, item ids and the epoch are not counted, and neither is any framing.
The accept or reject result of L3 and O2 is a boolean and counts 0 B. The messages are from each phase's
last timed run. A key update share that $S_j$ deals to itself never leaves $S_j$, so it is listed but
left out of the totals.

In [11]:
def sizes_text(sizes):
    return str(sizes[0]) if len(set(sizes)) == 1 else " / ".join(map(str, sizes))


wire_rows, message_lines, totals = [], [], {}
for phase, measured in PHASES.items():
    for name, direction, messages in measured.wire:
        sizes = [payload_size(message) for message in messages]
        wire_rows.append((phase, name, direction, len(messages), sizes_text(sizes), sum(sizes)))
        message_lines.append(f"{phase}, {name}, {direction}: {len(messages)} x {sizes_text(sizes)} B = {sum(sizes)} B")
        if direction != "local":
            totals.setdefault(phase, defaultdict(int))[direction] += sum(sizes)
print(table(("phase", "message", "direction", "count", "bytes each", "bytes total"), wire_rows))
print()
print(table(("phase", "U -> S total", "S -> U total", "S -> S total"), [
    (phase, f"{t['U -> S']} B", f"{t['S -> U']} B", f"{t['S -> S']} B") for phase, t in totals.items()
]))
print()
print("\n".join(message_lines))
print()
for phase, t in totals.items():
    print(f"{phase}: U -> S {t['U -> S']} B, S -> U {t['S -> U']} B, S -> S {t['S -> S']} B")

phase       message                           direction  count  bytes each      bytes total
----------  --------------------------------  ---------  -----  --------------  -----------
Register    ID_u (duplicate check)            U -> S     3      6               18
Register    TOPRFRequest                      U -> S     3      57350           172050
Register    TOPRFResponse                     S -> U     3      114688          344064
Register    RegisterKeyMessage                U -> S     3      3110            9330
Login       TOPRFRequest                      U -> S     2      57349           114698
Login       TOPRFResponse (with pk_u, sigma)  S -> U     2      117792          235584
Login       sid                               S -> U     2      32              64
Login       LoginProof                        U -> S     2      2457            4914
Outsource   OutsourceRequest                  U -> S     1      106709          106709
Retrieve    RetrieveRequest                  

## Conditions after the run, and the checks

The chain is timed again at the end and set next to every earlier $a_x$ series, with each series' minor
page faults and CPU clock. The last line gives how busy CPU 2 and its hyperthread sibling CPU 3 were over
the whole run. Other processes on the machine are not pinned away from either.
The cell fails the run unless every check above held.

In [12]:
chain_end = measure(a_x_trial)
load_after = os.getloadavg()
a_x_rows = [("start (building blocks)", chain)] + [(f"alternated with {phase}", PHASES[phase]) for phase in ("Register", "Login", "Outsource", "Retrieve")] + [("end", chain_end)]
print(table(("a_x measured", "median", "range (min to max)", "minor page faults per run", "CPU 2 MHz after a run"), [
    (name, duration(m.median("a_x")), spread(m.samples("a_x")), f"{m.median_faults('a_x'):.0f}", statistics.median(m.mhz)) for name, m in a_x_rows
]))
jiffies_after = cpu_jiffies()
busy = {cpu: 1 - (jiffies_after[cpu][1] - jiffies_before[cpu][1]) / (jiffies_after[cpu][0] - jiffies_before[cpu][0]) for cpu in (2, 3)}
print(f"busy share over the whole run: CPU 2 (the kernel) {busy[2]:.0%}, CPU 3 (its hyperthread sibling) {busy[3]:.0%}")
print(f"a_x at the end: median {in_ms(chain_end.median('a_x'))}, ratio end / start {chain_end.median('a_x') / A_X:.2f}")
print(f"load average before: {load_before}, after: {load_after}")
print(f"kernel main thread affinity: {sorted(os.sched_getaffinity(0))}")
all_measured = [chain, chain_end, *(m for m, _ in blocks.values()), *PHASES.values()]
failures = sum(m.failures for m in all_measured)
runs = sum(len(m.runs) + 1 for m in all_measured)
print(f"failures over all {runs} checked runs: {failures}")

EXPECTED_LIMITER = {
    ("Register", "U"): "chain", ("Register", "S_1"): "evaluate", ("Login", "U"): "chain", ("Login", "S_1"): "verify",
    ("Outsource", "U"): "chain", ("Outsource", "S_1"): "verify", ("Retrieve", "U"): "chain", ("Retrieve", "S_1"): "chain",
    ("Key update", "S_1"): "sign", ("Key update", "round"): "sign",
}
assert failures == 0, "every timed run returned a correct result"
assert all(from_code == run for from_code, run in counted.values()), "counted a_x calls equal the code-path chain counts"
assert limiters == EXPECTED_LIMITER, "each phase's limiter is the one the text names"
assert os.sched_getaffinity(0) == {2}, "still pinned to CPU 2"
print("all cost checks passed")

a_x measured               median  range (min to max)  minor page faults per run  CPU 2 MHz after a run
-------------------------  ------  ------------------  -------------------------  ---------------------
start (building blocks)    494 ms  494 ms to 502 ms    3510                       3900
alternated with Register   500 ms  467 ms to 506 ms    2471                       3929
alternated with Login      465 ms  462 ms to 477 ms    0                          4000
alternated with Outsource  459 ms  458 ms to 465 ms    0                          4000
alternated with Retrieve   495 ms  492 ms to 497 ms    3293                       3837
end                        493 ms  492 ms to 494 ms    3370                       3899
busy share over the whole run: CPU 2 (the kernel) 100%, CPU 3 (its hyperthread sibling) 0%
a_x at the end: median 493 ms, ratio end / start 1.00
load average before: (0.974609375, 0.8212890625, 0.83544921875), after: (1.03271484375, 0.919921875, 0.87451171875)
kernel ma

The $a_x$ series can differ by several percent between parts of the notebook. The table above sets each
series next to its page faults and its clock reading, so a reader can see which of them moved with it. The
clock reading is one coarse sample after each run, and no cycle counter is readable here without `perf`, so
the table narrows the cause without proving it. Pairing each phase run with the $a_x$ right after it keeps
this drift out of the share column.